# Qwen2.5-1.5B-Instruct Inference
Loading and running the Qwen/Qwen2.5-1.5B-Instruct model using 🤗 Transformers.

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define and create the directory path
drive_path = "/content/drive/MyDrive/msc/models"
os.makedirs(drive_path, exist_ok=True)

print(f"Directory is ready at: {drive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directory is ready at: /content/drive/MyDrive/msc/models


In [2]:
%pip install -q trl transformers datasets accelerate peft optuna bitsandbytes wandb
!pip install torchao -U

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,TrainerCallback
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
import optuna
import wandb
model_id = 'Qwen/Qwen2.5-1.5B-Instruct'

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Qwen uses eos_token as pad — required to avoid warnings
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded ✓')

Tokenizer loaded ✓


In [5]:
from datasets import load_dataset
import re

TRAINING_PATH = "/content/drive/MyDrive/Msc/data/final_combined_dataset_combined_instructions.jsonl"

# def contains_english_letters(text):
#     return bool(re.search('[a-zA-Z]', text))

def format_prompts(batch):
    formatted_texts = []
    for q, a in zip(batch["instruction"], batch["response_hebrew"]):
        # Use Qwen's native chat template
        messages = [
            {"role": "user",      "content": q},
            {"role": "assistant", "content": a},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

# Load → filter → split → format (correct order)
dataset = load_dataset("json", data_files=TRAINING_PATH, split="train")
# dataset = dataset.filter(lambda x: not contains_english_letters(x["answer_heb"]))
dataset = dataset.shuffle(seed=42)

split_dataset   = dataset.train_test_split(test_size=0.2, seed=42)
formatted_dataset = split_dataset.map(format_prompts, batched=True)

print(f"Train: {len(formatted_dataset['train'])} | Eval: {len(formatted_dataset['test'])}")


Train: 4298 | Eval: 1075


In [ ]:
from google.colab import userdata
import os
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

class LogLoraParamsCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        # Log the LoRA params for this trial to W&B
        wandb.config.update({
            "lora_r":       _current_lora_params.get("lora_r"),
            "lora_alpha":   _current_lora_params.get("lora_alpha"),
            "lora_dropout": _current_lora_params.get("lora_dropout"),
        })

    def on_train_end(self, args, state, control, **kwargs):
        wandb.finish()

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: talsomech (talsomech-bar-ilan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:

# QLoRA: 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # nf4 > fp4 for LLMs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,      # saves extra VRAM
)

_current_lora_params = {}
def my_hp_space(trial):
    _current_lora_params["lora_r"]       = trial.suggest_categorical("lora_r", [4, 8, 16])
    _current_lora_params["lora_alpha"]   = trial.suggest_categorical("lora_alpha", [8, 16, 32])
    _current_lora_params["lora_dropout"] = trial.suggest_float("lora_dropout", 0.05, 0.3)

    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.03, 0.10),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    }
def compute_objective(metrics):
    return metrics["eval_loss"]

def model_init(trial):
    r = _current_lora_params.get("lora_r", 8)
    alpha = _current_lora_params.get("lora_alpha", 16)
    dropout = _current_lora_params.get("lora_dropout", 0.05)
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    base_model = prepare_model_for_kbit_training(base_model)
    peft_config = LoraConfig(
         r=r,
         lora_alpha=alpha,
         lora_dropout=dropout,
         target_modules="all-linear",
         bias="none",
         task_type="CAUSAL_LM",
     )
    return get_peft_model(base_model, peft_config)


In [ ]:
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/Msc/models",
    max_steps=300,
    # max_seq_length=512,
    logging_steps=100,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=50,
    fp16=True,
    dataset_text_field="text",
    disable_tqdm=True,
    report_to="wandb",
    run_name="qwen-qlora-final"
)

trainer=SFTTrainer(
    model=model_init(None),
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
    peft_config=None,
    processing_class=tokenizer,
    callbacks=[LogLoraParamsCallback()],
    )
trainer.model_init=model_init

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Adding EOS to train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
best_run = trainer.hyperparameter_search(
    hp_space=my_hp_space,
    compute_objective=compute_objective,
    n_trials=10,                # Number of total tuning combinations to try
    direction="minimize",       # We want to minimize eval_loss
    backend="optuna",
)
print("Best run:", best_run)
print("Best hyperparameters:", best_run.hyperparameters)


[I 2026-05-30 14:59:08,276] A new study created in memory with name: no-name-7eaa8e92-8245-4672-824f-6fac55e01c29
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'eval_loss': '4.102', 'eval_runtime': '0.6234', 'eval_samples_per_second': '22.46', 'eval_steps_per_second': '3.208', 'eval_entropy': '3.288', 'eval_num_tokens': '3.031e+04', 'eval_mean_token_accuracy': '0.3782', 'epoch': '3.125'}
{'loss': '4.119', 'grad_norm': '2.924', 'learning_rate': '1.083e-06', 'entropy': '3.171', 'num_tokens': '6.01e+04', 'mean_token_accuracy': '0.4063', 'epoch': '6.25'}
{'eval_loss': '3.811', 'eval_runtime': '0.629', 'eval_samples_per_second': '22.26', 'eval_steps_per_second': '3.18', 'eval_entropy': '3.256', 'eval_num_tokens': '6.01e+04', 'eval_mean_token_accuracy': '0.4446', 'epoch': '6.25'}
{'eval_loss': '3.59', 'eval_runtime': '0.6506', 'eval_samples_per_second': '21.52', 'eval_steps_per_second': '3.074', 'eval_entropy': '3.227', 'eval_num_tokens': '8.958e+04', 'eval_mean_token_accuracy': '0.4479', 'epoch': '9.375'}
{'loss': '3.563', 'grad_norm': '2.175', 'learning_rate': '5.441e-07', 'entropy': '3.137', 'num_tokens': '1.193e+05', 'mean_token_accuracy': '0.

eval/entropy,█▆▃▂▁▁
eval/loss,█▅▃▂▁▁
eval/mean_token_accuracy,▁▆▇███
eval/num_tokens,▁▂▄▅▇█
eval/runtime,▂▃█▁▂▁
eval/samples_per_second,▇▅▁█▇█
eval/steps_per_second,▇▅▁▇▇█
train/entropy,█▄▁
train/epoch,▁▂▂▄▅▅▇███
train/global_step,▁▂▂▄▅▅▇███
+5,...


...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'eval_loss': '2.343', 'eval_runtime': '0.6334', 'eval_samples_per_second': '22.1', 'eval_steps_per_second': '3.157', 'eval_entropy': '2.587', 'eval_num_tokens': '1.335e+06', 'eval_mean_token_accuracy': '0.5802', 'epoch': '3.125'}
{'loss': '2.453', 'grad_norm': '3.202', 'learning_rate': '1.138e-05', 'entropy': '2.638', 'num_tokens': '1.365e+06', 'mean_token_accuracy': '0.5854', 'epoch': '6.25'}
{'eval_loss': '1.842', 'eval_runtime': '0.6175', 'eval_samples_per_second': '22.67', 'eval_steps_per_second': '3.239', 'eval_entropy': '2.306', 'eval_num_tokens': '1.365e+06', 'eval_mean_token_accuracy': '0.6558', 'epoch': '6.25'}
{'eval_loss': '1.769', 'eval_runtime': '0.6283', 'eval_samples_per_second': '22.28', 'eval_steps_per_second': '3.183', 'eval_entropy': '2.2', 'eval_num_tokens': '1.394e+06', 'eval_mean_token_accuracy': '0.6677', 'epoch': '9.375'}
{'loss': '1.512', 'grad_norm': '4.658', 'learning_rate': '5.717e-06', 'entropy': '2.06', 'num_tokens': '1.424e+06', 'mean_token_accuracy': '0

eval/entropy,█▄▃▂▁▁
eval/loss,█▂▂▁▁▁
eval/mean_token_accuracy,▁▇████
eval/num_tokens,▁▂▄▅▇█
eval/runtime,█▁▆▃▁▄
eval/samples_per_second,▁█▃▆█▅
eval/steps_per_second,▁█▃▆█▅
train/entropy,█▂▁
train/epoch,▁▂▂▄▅▅▇███
train/global_step,▁▂▂▄▅▅▇███
+5,...


[I 2026-05-30 15:45:41,053] Trial 9 finished with value: 1.722070336341858 and parameters: {'lora_r': 4, 'lora_alpha': 32, 'lora_dropout': 0.08570320022594267, 'learning_rate': 1.698030404080942e-05, 'warmup_ratio': 0.06301689089443885, 'weight_decay': 0.03498608886967613}. Best is trial 8 with value: 1.70637047290802.


Best run: BestRun(run_id='8', objective=1.70637047290802, hyperparameters={'lora_r': 4, 'lora_alpha': 16, 'lora_dropout': 0.13482101379864803, 'learning_rate': 3.48576351528276e-05, 'warmup_ratio': 0.06349163207497396, 'weight_decay': 0.024825698786720952}, run_summary=None)
Best hyperparameters: {'lora_r': 4, 'lora_alpha': 16, 'lora_dropout': 0.13482101379864803, 'learning_rate': 3.48576351528276e-05, 'warmup_ratio': 0.06349163207497396, 'weight_decay': 0.024825698786720952}


In [ ]:
Best hyperparameters: {'lora_r': 8, 'lora_alpha': 32, 'lora_dropout': 0.14527575281065663, 'learning_rate': 1.5520566449238817e-05, 'warmup_ratio': 0.034256172553320086, 'weight_decay': 0.005597585923559124}

In [ ]:
wandb.init()

In [ ]:
# Apply best hyperparameters
for param, value in best_run.hyperparameters.items():
    setattr(trainer.args, param, value)

# Re-init model with the best LoRA params (stored in _current_lora_params)
_current_lora_params["lora_r"]       = best_run.hyperparameters["lora_r"]
_current_lora_params["lora_alpha"]   = best_run.hyperparameters["lora_alpha"]
_current_lora_params["lora_dropout"] = best_run.hyperparameters["lora_dropout"]
trainer.model = model_init(None)
trainer.args.max_steps = 300
# trainer.train(resume_from_checkpoint=True)
# trainer.train()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
# Your selected best hyperparameters (with r=16 to train ~1.18% of params)
final_hyperparameters = {
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.13482101379864803,
    'learning_rate': 3.48576351528276e-05,
    'warmup_ratio': 0.06349163207497396,
    'weight_decay': 0.024825698786720952
}

# 1. Fresh SFTConfig with WandB disabled and 8-bit optimizer
final_training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/Msc/models/Qwen-FT-final",
    # max_steps=1000,
    num_train_epochs=3,

    logging_steps=100,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    # fp16=True,
    # optim="paged_adamw_8bit",  # Added to save memory and avoid scaler issues
    dataset_text_field="text",
    report_to="none",  # Explicitly disable wandb
    learning_rate=final_hyperparameters['learning_rate'],
    warmup_ratio=final_hyperparameters['warmup_ratio'],
    weight_decay=final_hyperparameters['weight_decay'],
    bf16=torch.cuda.is_available(),
    per_device_train_batch_size=2,   # The number of samples processed at once per GPU. (Try 1 or 2).
    gradient_accumulation_steps=4,   # Number of steps to accumulate gradients before updating weights.
    gradient_checkpointing=True,     # Saves VRAM by discarding intermediate activations and recomputing them.
    fp16=False,
    optim="adamw_torch",
)

# 2. Fresh LoRA Config
final_peft_config = LoraConfig(
    r=final_hyperparameters['lora_r'],
    lora_alpha=final_hyperparameters['lora_alpha'],
    lora_dropout=final_hyperparameters['lora_dropout'],
    # target_modules="all-linear",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
       ],
    bias="none",
    task_type="CAUSAL_LM",
)

# 3. Fresh Model Initialization
print("Loading base model in float16...")
final_base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    # quantization_config=bnb_config,
    # device_map="auto",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
    # torch_dtype=torch.float16,  # Force float16 to prevent BFloat16 scaler errors
)
# final_base_model = prepare_model_for_kbit_training(final_base_model)
final_model = get_peft_model(final_base_model, final_peft_config)

# count_lora_parameters(final_model)

# 4. Fresh SFTTrainer
final_trainer = SFTTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
    processing_class=tokenizer,
)

# 5. Train!
print("\nStarting final training with new SFTTrainer...")
final_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading base model in float16...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting final training with new SFTTrainer...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.755642,2.321977,2.804700,186565.000000,0.554874
200,2.249695,2.207274,2.634746,386720.000000,0.566722
300,2.186618,2.159876,2.531057,578868.000000,0.571824
400,2.185562,2.128715,2.451853,763848.000000,0.574975
500,2.087912,2.104875,2.436444,962020.000000,0.577828
600,2.057840,2.087673,2.342652,1154850.000000,0.579689
700,2.021160,2.072461,2.357618,1347684.000000,0.583178
800,2.042807,2.060191,2.305637,1549397.000000,0.583507
900,1.985303,2.050349,2.316185,1744945.000000,0.585630
1000,1.993432,2.043664,2.285428,1927176.000000,0.585799


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.755642,2.321977,2.804700,186565.000000,0.554874
200,2.249695,2.207274,2.634746,386720.000000,0.566722
300,2.186618,2.159876,2.531057,578868.000000,0.571824
400,2.185562,2.128715,2.451853,763848.000000,0.574975
500,2.087912,2.104875,2.436444,962020.000000,0.577828
600,2.057840,2.087673,2.342652,1154850.000000,0.579689
700,2.021160,2.072461,2.357618,1347684.000000,0.583178
800,2.042807,2.060191,2.305637,1549397.000000,0.583507
900,1.985303,2.050349,2.316185,1744945.000000,0.585630
1000,1.993432,2.043664,2.285428,1927176.000000,0.585799


In [8]:
save_path = "/content/drive/MyDrive/Msc/models/Qwen-FT-best4"
final_trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved to: {save_path}")


Saved to: /content/drive/MyDrive/Msc/models/Qwen-FT-best3


In [ ]:
def count_lora_parameters(model):
    trainable_params, all_param = model.get_nb_trainable_parameters()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.4f}"
    )

# Check the parameters of the loaded LoRA model
count_lora_parameters(trainer.model)

trainable params: 9232384 || all params: 1552946688 || trainable%: 0.5945


### Model Inference
Load the fine-tuned model from Google Drive and test it with a sample question.

In [9]:
from peft import PeftModel
import torch

# Path where the model was saved
model_save_path = "/content/drive/MyDrive/Msc/models/Qwen-FT-best4"

# Load the base model again (using 4-bit for efficiency)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    # quantization_config=bnb_config,
    device_map="auto",
)

# Load the fine-tuned LoRA weights
model = PeftModel.from_pretrained(base_model, model_save_path)
model.eval()

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_save_path)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [10]:
def ask_qwen(question):
    messages = [
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            # temperature=0.7,
            # top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

    # Trim the input tokens from the response
    response_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]
    return response

# Test it out!
question = "why are the sky blue?"
print(f"Question: {question}")
print(f"Answer: {ask_qwen(question)}")

Question: why are the sky blue?
Answer: השמיים ידועים כ-transparentים וירחנים על ידי גוף השמש שמקושר לתוך האטמוספירה. הפליאה של אור השמש ב trail, המכונה פליאה או ראייה, מונעת את אור החום האזורי. כאשר אור השמש נופל על פני כדור הארץ, חלק מהאור מגיע לכדור הארץ, וחלק נופל לאטמוספירה. חלק מהאור הזה נופל לתוך המים והחומר הלא ספציפי בתוכם, כמו אבק, ולצורך האטמוספירה, והוא מתפרק למחזור חומרי אטמוספרתי. חלק מהאור_remaining מגיע למכונית ומסתובב למעלה, מתרחב ומתפשף. חלק מהאור הזה מגיע לקצה האטמוספירה, ונקרא 'פליאה'. הפלייה הזו מספקת תוצרת זריזה, וניתן לראות אותה בצורת רוחב, וסימונים להיפך. הפליאה הזו עוזרת להגביר את ההבדלים בין הצבעים הנמוכים יותר לרוב, כמו צבעים ברוקים ואפורים, לבין הצבעים גבוהים יותר, כמו כחול, אפור וירח. אז, זה מה שהופך את השמים לבורחנים וירחנים.
